In [7]:
# Load required libraries
import pandas as pd
from pyhere import here
import statsmodels.api as sm
import yaml

In [8]:
# Construct the path to the CSV file, relative to the root of the Git repository
csv_path = here("data", "raw", "auto.csv")

# Load the data
auto_data = pd.read_csv(csv_path)

In [9]:
# List the first 3 observations
print(auto_data.head(3))

          make  price  mpg  rep78  headroom  trunk  weight  length  turn  \
0  AMC Concord   4099   22    3.0       2.5     11    2930     186    40   
1    AMC Pacer   4749   17    3.0       3.0     11    3350     173    40   
2   AMC Spirit   3799   22    NaN       3.0     12    2640     168    35   

   displacement  gear_ratio   foreign  
0           121        3.58  Domestic  
1           258        2.53  Domestic  
2           121        3.08  Domestic  


In [10]:
# Calculate descriptive statistics like the mean
average_mpg = auto_data['mpg'].mean()
print(f"Average MPG: {average_mpg}")

Average MPG: 21.2972972972973


In [11]:
# Run an OLS regression
auto_data['foreign'] = auto_data['foreign'].apply(lambda x: 1 if x == 'Foreign' else 0)
auto_data = sm.add_constant(auto_data)
ols_model = sm.OLS(auto_data['price'], auto_data[['const', 'mpg', 'foreign']]).fit()

print(ols_model.summary())

slope_mpg = ols_model.params['mpg']
se_mpg = ols_model.bse['mpg']
print(f"Coefficient and Standard Error for mpg: {slope_mpg} ({se_mpg})")

slope_foreign = ols_model.params['foreign']
se_foreign = ols_model.bse['foreign']
print(f"Coefficient and Standard Error for foreign: {slope_foreign} ({se_foreign})")

slope_intercept = ols_model.params['const']
se_intercept = ols_model.bse['const']
print(f"Coefficient and Standard Error for intercept: {slope_intercept} ({se_intercept})")

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.284
Model:                            OLS   Adj. R-squared:                  0.264
Method:                 Least Squares   F-statistic:                     14.07
Date:                Wed, 14 Jan 2026   Prob (F-statistic):           7.12e-06
Time:                        20:59:38   Log-Likelihood:                -683.36
No. Observations:                  74   AIC:                             1373.
Df Residuals:                      71   BIC:                             1380.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.191e+04   1158.634     10.275      0.0

In [12]:
# Write to results/example.yaml
result_list = {
    "average_mpg": float(average_mpg),
    "ols_slope_mpg": float(slope_mpg),
}
results_path = here("results", "example.yaml")
with open(results_path, "w") as yaml_file:
    yaml.dump(result_list, yaml_file)
print("Results written to results/example.yaml")

Results written to results/example.yaml
